# Customer analysis - 100K rows


The file is only ~17mb so it is fine to just load the whole thing with pandas.

In [1]:
import sys
import time
import resource
import pandas as pd
from IPython.display import display

FILE = "data/customers-100000.csv"

pd.set_option("display.max_rows", 200)


def peak_mem_mb():
    m = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # mac returns bytes here but linux returns kilobytes, so check the platform
    if sys.platform == "darwin":
        return m / 1024.0 / 1024.0
    return m / 1024.0


def as_table(counts, n=None, total=None, name="count"):
    # a value_counts() result -> small table with a percentage column
    denom = total if total is not None else counts.sum()
    out = (counts if n is None else counts.head(n)).rename(name).to_frame()
    out["% of rows"] = (out[name] / denom * 100).round(2)
    return out


start = time.time()

## Load the file

In [2]:
df = pd.read_csv(FILE, parse_dates=["Subscription Date"])
rows = len(df)

display(pd.Series({
    "file": FILE,
    "rows": rows,
    "columns": len(df.columns),
    "in memory (mb)": round(df.memory_usage(deep=True).sum() / 1024 ** 2, 1),
}, name="value").to_frame())

df.head()

,value
file,customers-100000.csv
rows,100000
columns,12
in memory (mb),69.7


,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,ffeCAb7AbcB0f07,Jared,Jarvis,Sanchez-Fletcher,Hatfieldshire,Eritrea,274.188.8773x41185,001-215-760-4642x969,gabriellehartman@benjamin.com,2021-11-11,https://www.mccarthy.info/
1,2,b687FfC4F1600eC,Marie,Malone,Mckay PLC,Robertsonburgh,Botswana,283-236-9529,(189)129-8356x63741,kstafford@sexton.com,2021-05-14,http://www.reynolds.com/
2,3,9FF9ACbc69dcF9c,Elijah,Barrera,Marks and Sons,Kimbury,Barbados,8252703789,459-916-7241x0909,jeanettecross@brown.com,2021-03-17,https://neal.com/
3,4,b49edDB1295FF6E,Sheryl,Montgomery,"Kirby, Vaughn and Sanders",Briannaview,Antarctica (the territory South of 60 deg S),425.475.3586,(392)819-9063,thomassierra@barrett.com,2020-09-23,https://www.powell-bryan.com/
4,5,3dcCbFEB17CCf2E,Jeremy,Houston,Lester-Manning,South Brianna,Micronesia,+1-223-666-5313x4530,252-488-3850x692,rubenwatkins@jacobs-wallace.info,2020-09-18,https://www.carrillo.com/


## Data quality

In [3]:
quality = pd.DataFrame({
    "missing": df.isnull().sum(),
    "blank strings": {
        col: (df[col].astype(str).str.strip() == "").sum()
        for col in df.columns if df[col].dtype == object
    },
}).fillna(0).astype(int)

print(f"{quality['missing'].sum()} missing values, {quality['blank strings'].sum()} blank strings")
quality

0 missing values, 0 blank strings


,missing,blank strings
City,0,0
Company,0,0
Country,0,0
Customer Id,0,0
Email,0,0
First Name,0,0
Index,0,0
Last Name,0,0
Phone 1,0,0
Phone 2,0,0


Unique values per column.

In [4]:
uniques = df.nunique().rename("unique values").to_frame()
uniques["% of rows"] = (uniques["unique values"] / rows * 100).round(1)
uniques

,unique values,% of rows
Index,100000,100.0
Customer Id,100000,100.0
First Name,690,0.7
Last Name,1000,1.0
Company,71994,72.0
City,49154,49.2
Country,243,0.2
Phone 1,100000,100.0
Phone 2,100000,100.0
Email,99995,100.0


Customer Id and Email are supposed to identify one person, so check them.

In [5]:
dup_id = rows - df["Customer Id"].nunique()
dup_email = rows - df["Email"].nunique()

display(pd.Series({
    "duplicated Customer Id": dup_id,
    "duplicated Email": dup_email,
}, name="rows").to_frame())

if dup_email > 0:
    print("the customers sharing an email:")
    cols = ["Customer Id", "First Name", "Last Name", "Country", "Email"]
    display(df[df.duplicated("Email", keep=False)].sort_values("Email")[cols])

,rows
duplicated Customer Id,0
duplicated Email,5


the customers sharing an email:


,Customer Id,First Name,Last Name,Country,Email
85112,5BcEBD82eBFf102,Charles,Freeman,American Samoa,imitchell@church.com
93818,DDdA80d1beD99b6,Joshua,Gallegos,El Salvador,imitchell@church.com
90751,8A2E9e5af981CBd,Alvin,Stephens,Liechtenstein,julia03@briggs.com
96681,64ff4a97cCF99Da,Kathy,Rojas,Ghana,julia03@briggs.com
18196,EdAD6f6E81F68Fc,Alison,Archer,Central African Republic,kwalls@white.com
95616,d44BCA84DEAf5EB,Lindsey,Carson,Dominica,kwalls@white.com
31997,B29Fb3EF18604f8,Jaclyn,Torres,Kyrgyz Republic,ushields@saunders.com
73003,C2DB1665c793Df8,Cristina,Cox,Montenegro,ushields@saunders.com
30556,b0E5A8E4E2beF72,Tammy,Ramirez,Sweden,vgeorge@mendoza.com
71528,b6Bcf35CE3e63b6,Kenneth,Mcpherson,Finland,vgeorge@mendoza.com


## Where the customers are

In [6]:
countries = df["Country"].value_counts()

display(as_table(countries, 10, rows).style.set_caption("top 10 countries"))
display(as_table(countries.tail(5), total=rows).style.set_caption("bottom 5 countries"))

pd.Series({
    "countries": len(countries),
    "top 10 share of all customers (%)": round(countries.head(10).sum() * 100 / rows, 1),
    "average per country": round(rows / len(countries)),
    "biggest country": countries.max(),
    "smallest country": countries.min(),
}, name="value").to_frame()

,count,% of rows
Congo,835,0.840000
Korea,820,0.820000
Saudi Arabia,463,0.460000
Pitcairn Islands,456,0.460000
Saint Martin,453,0.450000
Paraguay,445,0.440000
Canada,444,0.440000
American Samoa,443,0.440000
Saint Kitts and Nevis,443,0.440000
Cook Islands,441,0.440000


,count,% of rows
Moldova,371,0.370000
Jordan,365,0.360000
Saint Lucia,365,0.360000
Slovenia,361,0.360000
Greece,359,0.360000


,value
countries,243.0
top 10 share of all customers (%),5.2
average per country,412.0
biggest country,835.0
smallest country,359.0


In [7]:
cities = df["City"].value_counts()
print(f"{len(cities)} cities")
as_table(cities, 10, rows)

49154 cities


,count,% of rows
Lake Frederick,16,0.02
West Alec,15,0.02
East Jeremy,15,0.02
East Lee,15,0.02
New Christopher,15,0.02
Lake Alexander,14,0.01
West Bailey,14,0.01
New Wayne,14,0.01
Port Brandy,13,0.01
New Terrance,13,0.01


## Subscriptions over time

In [8]:
dates = df["Subscription Date"]

pd.Series({
    "first signup": dates.min().date(),
    "last signup": dates.max().date(),
    "distinct days": dates.nunique(),
}, name="value").to_frame()

,value
first signup,2020-01-01
last signup,2022-05-29
distinct days,880


In [9]:
per_year = dates.dt.year.value_counts().sort_index()
display(as_table(per_year, total=rows, name="signups"))

# 2022 is not a full year so only compare the months we actually have
if 2021 in per_year.index and 2020 in per_year.index:
    growth = (per_year[2021] - per_year[2020]) * 100.0 / per_year[2020]
    print(f"2020 -> 2021 change: {growth:+.2f}%")

,signups,% of rows
2020,41898,41.90
2021,41211,41.21
2022,16891,16.89


2020 -> 2021 change: -1.64%


In [10]:
per_month = dates.dt.to_period("M").value_counts().sort_index()
print(f"{len(per_month)} months, average {per_month.mean():.0f} signups/month")

display(pd.Series({
    f"best month ({per_month.idxmax()})": per_month.max(),
    f"worst month ({per_month.idxmin()})": per_month.min(),
}, name="signups").to_frame())

as_table(per_month, total=rows, name="signups")

29 months, average 3448 signups/month


,signups
best month (2020-12),3616
worst month (2021-02),3106


,signups,% of rows
2020-01,3557,3.56
2020-02,3271,3.27
2020-03,3530,3.53
2020-04,3431,3.43
2020-05,3500,3.50
2020-06,3540,3.54
2020-07,3516,3.52
2020-08,3552,3.55
2020-09,3504,3.50
2020-10,3481,3.48


In [11]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
per_day = dates.dt.day_name().value_counts().reindex(order)
as_table(per_day, total=rows, name="signups")

,signups,% of rows
Monday,14225,14.22
Tuesday,14174,14.17
Wednesday,14147,14.15
Thursday,14325,14.32
Friday,14417,14.42
Saturday,14383,14.38
Sunday,14329,14.33


## Email / company / website

In [12]:
domains = df["Email"].str.split("@").str[1].str.lower()
dom_counts = domains.value_counts()

print(f"{len(dom_counts)} distinct email domains, "
      f"{(dom_counts == 1).sum()} of them used by a single customer "
      f"({(dom_counts == 1).mean():.1%} of domains)")

display(as_table(dom_counts, 10, rows).style.set_caption("top 10 email domains"))

tld = domains.str.split(".").str[-1]
as_table(tld.value_counts(), 8, rows)

38322 distinct email domains, 33138 of them used by a single customer (86.5% of domains)


,count,% of rows
mckee.com,59,0.060000
pugh.com,59,0.060000
terrell.com,57,0.060000
lawrence.com,57,0.060000
gaines.com,56,0.060000
romero.com,56,0.060000
delgado.com,56,0.060000
glass.com,55,0.060000
jacobson.com,55,0.060000
fitzpatrick.com,55,0.060000


,count,% of rows
com,60120,60.12
org,10004,10.00
info,9988,9.99
net,9978,9.98
biz,9910,9.91


In [13]:
companies = df["Company"].value_counts()
print(f"{len(companies)} distinct companies")
as_table(companies, 10, rows)

71994 distinct companies


,count,% of rows
Wilkerson Ltd,17,0.02
Campbell Ltd,17,0.02
Acosta Ltd,16,0.02
Booker and Sons,16,0.02
Mckenzie PLC,15,0.02
Mccarty and Sons,15,0.02
Gregory Group,15,0.02
Farmer Ltd,14,0.01
Riggs PLC,14,0.01
Gomez Inc,14,0.01


A real company email usually matches the company website, check if it does.

In [14]:
site = df["Website"].str.replace("https://", "", regex=False)
site = site.str.replace("http://", "", regex=False)
site = site.str.replace("www.", "", regex=False).str.strip("/").str.lower()

same = (site == domains).sum()
print(f"email domain == website domain: {same} rows ({same / rows:.2%})")

email domain == website domain: 19 rows (0.02%)


## Names

In [15]:
first = df["First Name"].value_counts()
last = df["Last Name"].value_counts()
print(f"{len(first)} distinct first names, {len(last)} distinct last names")

display(as_table(first, 10, rows).style.set_caption("top 10 first names"))
display(as_table(last, 10, rows).style.set_caption("top 10 last names"))

690 distinct first names, 1000 distinct last names


,count,% of rows
Joan,183,0.180000
Audrey,182,0.180000
Bridget,182,0.180000
Anne,180,0.180000
Melinda,177,0.180000
Selena,176,0.180000
Benjamin,175,0.180000
Lee,175,0.180000
Diana,174,0.170000
Kaitlyn,174,0.170000


,count,% of rows
Campbell,139,0.140000
Carney,132,0.130000
Gardner,131,0.130000
Patterson,130,0.130000
Middleton,127,0.130000
Cisneros,127,0.130000
Matthews,127,0.130000
Tate,126,0.130000
Rubio,126,0.130000
Doyle,125,0.120000


In [16]:
full = df["First Name"] + " " + df["Last Name"]
print(f"people sharing the exact same full name: {rows - full.nunique()}")
as_table(full.value_counts(), 5, rows)

people sharing the exact same full name: 7056


,count,% of rows
Jesse Gordon,4,0.0
Jeanette Ruiz,4,0.0
Connor Harris,4,0.0
Marisa Golden,4,0.0
Hayden Watts,4,0.0


## Phone numbers

In [17]:
p1 = df["Phone 1"]

phones = pd.DataFrame(
    {"rows": [p1.str.contains("x").sum(), p1.str.startswith("+").sum(), (p1 == df["Phone 2"]).sum()]},
    index=["Phone 1 has an extension (x)", "Phone 1 has a country code (+)", "Phone 1 == Phone 2"],
)
phones["% of rows"] = (phones["rows"] / rows * 100).round(1)
display(phones)

digits = p1.str.replace(r"\D", "", regex=True).str.len()
digits.describe()[["min", "max", "mean"]].round(1).rename("digits in Phone 1").to_frame()

,rows,% of rows
Phone 1 has an extension (x),59954,60.0
Phone 1 has a country code (+),15951,16.0
Phone 1 == Phone 2,0,0.0


,digits in Phone 1
min,10.0
max,18.0
mean,13.0


## Done

In [18]:
print(f"done in {time.time() - start:.1f} seconds, peak memory {peak_mem_mb():.0f} mb")

done in 3.9 seconds, peak memory 259 mb
